## MODELO PREDICTIVO DE BROTES DE DENGUE POR VARIABLES CLIMATICAS - MINSA
### Notebook 1: Pipeline de datos (raw → Bronze → Silver → Gold)

En notebook tenemos: 
- Configuración
- Descarga del clima
- Bronze
- Silver dengue
- Silver Clima
- Gold
- Tiempos
- Escalabilidad

In [1]:
import urllib.request
import pandas as pd
import time
import csv
import json
from contextlib import contextmanager
from datetime import date, timedelta
from pathlib import Path
from urllib.parse import urlencode

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

# Spark usa 200 por defecto - se debe correr ambos valores
SHUFFLE_PARTITIONS = 8
USAR_CACHE = True

ANIO_INICIO, ANIO_FIN = 2000, 2024

spark = (
    SparkSession.builder
    .appName("ModeloPredictivoDenguePeru")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)
    .getOrCreate()
)
print(f"Spark {spark.version} | shuffle partitions = {SHUFFLE_PARTITIONS} | cache = {USAR_CACHE}")

def resolver_ruta_datos(nombre_archivo):
    candidatos = [
        Path.cwd() / "data" / "raw" / nombre_archivo,
        Path.cwd().parent / "data" / "raw" / nombre_archivo,
        Path("/home/jovyan/work/data/raw") / nombre_archivo,
    ]
    for candidato in candidatos:
        if candidato.exists():
            return str(candidato)
    revisadas = "\n".join(f"- {ruta}" for ruta in candidatos)
    raise FileNotFoundError(f"No se encontró {nombre_archivo}. Colócalo en data/raw.\n{revisadas}")

def ruta_salida(capa, dataset):
    base = Path.cwd()
    if base.name == "notebooks":
        base = base.parent
    return str(base / "data" / capa / dataset)

tiempos = []
# Mide una etapa
@contextmanager
def cronometro(etapa):
    inicio = time.perf_counter()
    yield
    duracion = time.perf_counter() - inicio
    tiempos.append({"etapa": etapa, "segundos": round(duracion, 2)})
    print(f"[{etapa}] {duracion:.2f} s")

# Tener en cuenta que las semanas epideomiologicas empiezan el domingo
def inicio_anio_epi(anio):
    d = date(anio, 1, 4)
    return d - timedelta(days=(d.weekday() + 1) % 7)

ANIOS_53 = [a for a in range(ANIO_INICIO, ANIO_FIN + 1)
            if (inicio_anio_epi(a + 1) - inicio_anio_epi(a)).days == 371]
print("Años epidemiológicos con 53 semanas:", ANIOS_53)

Spark 3.5.0 | shuffle partitions = 8 | cache = True
Años epidemiológicos con 53 semanas: [2003, 2008, 2014, 2020]


### Descarga del clima (NASA POWER → data/raw/clima_nasa)
Un punto por departamento modelado, en el foco de dengue. Criterios de elección de celda:
1. mover solo si la celda difiere > ~500 m de la ciudad
2. en costa, celda costera del oeste
3. en selva, vecina de la misma cuenca

In [2]:
URL_BASE = "https://power.larc.nasa.gov/api/temporal/daily/point"
dir_clima = Path(ruta_salida("raw", "clima_nasa"))
dir_clima.mkdir(parents=True, exist_ok=True)

PUNTOS_CLIMA = {
    # cod: (departamento, foco / celda elegida, lat, lon)
    "01": ("AMAZONAS",      "Bagua (central)",          -5.64,  -78.53),
    "02": ("ANCASH",        "Chimbote (central)",       -9.07,  -78.59),
    "05": ("AYACUCHO",      "Llochegua → NE valle Ene", -11.91, -73.285),
    "06": ("CAJAMARCA",     "Jaén (central)",           -5.71,  -78.81),
    "07": ("CALLAO",        "Ventanilla → O costera",   -11.87, -77.755),
    "08": ("CUSCO",         "Quillabamba → NO Urubamba",-12.36, -73.315),
    "10": ("HUANUCO",       "Tingo María → NE Huallaga",-8.80,  -75.375),
    "11": ("ICA",           "Ica → NO Chincha",         -13.57, -76.355),
    "12": ("JUNIN",         "Chanchamayo → E Perené",   -11.05, -74.695),
    "13": ("LA LIBERTAD",   "Trujillo → O costera",     -8.11,  -79.655),
    "14": ("LAMBAYEQUE",    "Chiclayo (central)",       -6.77,  -79.84),
    "15": ("LIMA",          "Lima norte → O costera",   -11.97, -77.655),
    "16": ("LORETO",        "Iquitos (central)",        -3.75,  -73.25),
    "17": ("MADRE DE DIOS", "Puerto Maldonado (central)",-12.59, -69.19),
    "19": ("PASCO",         "Constitución (central)",   -9.86,  -75.01),
    "20": ("PIURA",         "Piura (central)",          -5.19,  -80.63),
    "22": ("SAN MARTIN",    "Tarapoto (central)",       -6.49,  -76.36),
    "24": ("TUMBES",        "Tumbes (central)",         -3.57,  -80.45),
    "25": ("UCAYALI",       "Pucallpa (central)",       -8.38,  -74.55),
}

def nombre_archivo(cod, lat, lon):
    return f"clima_{cod}_{lat}_{lon}.json"   # las coordenadas forman parte de la clave del caché

def descargar_punto(cod, lat, lon):
    destino = dir_clima / nombre_archivo(cod, lat, lon)
    if destino.exists():
        return destino
    consulta = urlencode({
        "parameters": "T2M,RH2M,PRECTOTCORR", "community": "AG",
        "latitude": lat, "longitude": lon,
        "start": "20000101", "end": "20241231", "format": "JSON",
    })
    with urllib.request.urlopen(f"{URL_BASE}?{consulta}", timeout=180) as resp:
        destino.write_bytes(resp.read())
    time.sleep(2)
    return destino

# Mover a obsoleto/ los JSON que ya no corresponden a ningún punto vigente
vigentes = {nombre_archivo(c, la, lo) for c, (_, _, la, lo) in PUNTOS_CLIMA.items()}
dir_obsoleto = dir_clima / "obsoleto"
dir_obsoleto.mkdir(exist_ok=True)
for archivo in dir_clima.glob("clima_*.json"):
    if archivo.name not in vigentes:
        archivo.rename(dir_obsoleto / archivo.name)

filas = []
for cod, (dep, ref, lat, lon) in PUNTOS_CLIMA.items():
    datos = json.loads(descargar_punto(cod, lat, lon).read_text())
    elevacion = datos["geometry"]["coordinates"][2]
    p = datos["properties"]["parameter"]
    n_relleno = sum(v == -999 for serie in p.values() for v in serie.values())
    print(f"{cod} {dep:<14} {ref:<27} elev={elevacion:>7.1f} m  días={len(p['T2M'])}  -999={n_relleno}")
    for fecha, t2m in p["T2M"].items():
        filas.append((cod, dep, fecha, t2m, p["RH2M"][fecha], p["PRECTOTCORR"][fecha]))

ruta_clima_csv = dir_clima / "clima_departamentos.csv"
with open(ruta_clima_csv, "w", newline="", encoding="utf-8") as f:
    escritor = csv.writer(f)
    escritor.writerow(["cod_dep", "departamento", "fecha", "T2M", "RH2M", "PRECTOTCORR"])
    escritor.writerows(filas)
print(f"\n{len(filas)} filas escritas en {ruta_clima_csv}")

01 AMAZONAS       Bagua (central)             elev= 1312.2 m  días=9132  -999=0
02 ANCASH         Chimbote (central)          elev=  105.7 m  días=9132  -999=0
05 AYACUCHO       Llochegua → NE valle Ene    elev=  914.6 m  días=9132  -999=0
06 CAJAMARCA      Jaén (central)              elev= 1312.2 m  días=9132  -999=0
07 CALLAO         Ventanilla → O costera      elev=   59.4 m  días=9132  -999=0
08 CUSCO          Quillabamba → NO Urubamba   elev= 1411.6 m  días=9132  -999=0
10 HUANUCO        Tingo María → NE Huallaga   elev=  684.2 m  días=9132  -999=0
11 ICA            Ica → NO Chincha            elev=  483.0 m  días=9132  -999=0
12 JUNIN          Chanchamayo → E Perené      elev= 1767.2 m  días=9132  -999=0
13 LA LIBERTAD    Trujillo → O costera        elev=  147.7 m  días=9132  -999=0
14 LAMBAYEQUE     Chiclayo (central)          elev=   33.9 m  días=9132  -999=0
15 LIMA           Lima norte → O costera      elev=   59.4 m  días=9132  -999=0
16 LORETO         Iquitos (central)     

### 2. Ingesta Bronze (dato tal cual, todo texto, con linaje)

In [3]:
def ingestar_bronze(ruta_csv, destino, sep, requeridas):
    df = spark.read.csv(str(ruta_csv), header=True, inferSchema=False, sep=sep)
    faltantes = set(requeridas) - set(df.columns)
    if faltantes:
        raise ValueError(f"{Path(ruta_csv).name}: faltan columnas {sorted(faltantes)}")
    (df.withColumn("_archivo_origen", F.lit(Path(ruta_csv).name))
       .withColumn("_fecha_ingesta", F.current_timestamp())
       .write.mode("overwrite").parquet(destino))

with cronometro("bronze_dengue"):
    ingestar_bronze(resolver_ruta_datos("datos_abiertos_vigilancia_dengue_2000_2024.csv"),
                    ruta_salida("bronze", "dengue"), ";",
                    ["departamento", "provincia", "distrito", "ubigeo", "ano", "semana",
                     "diagnostic", "edad", "tipo_edad", "sexo"])

with cronometro("bronze_clima"):
    ingestar_bronze(dir_clima / "clima_departamentos.csv", ruta_salida("bronze", "clima"), ",",
                    ["cod_dep", "departamento", "fecha", "T2M", "RH2M", "PRECTOTCORR"])

[bronze_dengue] 12.13 s
[bronze_clima] 2.59 s


### 3. Silver: dengue (tipos declarados y reglas de calidad)

In [4]:
# Tipa, normaliza y evalúa reglas
def transformar_dengue_silver(df_bronze):
    df = (
        df_bronze.select(
            F.upper(F.trim("departamento")).alias("departamento"),
            F.upper(F.trim("provincia")).alias("provincia"),
            F.upper(F.trim("distrito")).alias("distrito"),
            F.lpad(F.trim("ubigeo"), 6, "0").alias("ubigeo"),
            F.col("ano").cast("int").alias("ano"),
            F.col("semana").cast("int").alias("semana"),
            F.trim("diagnostic").alias("diagnostic"),
            F.col("edad").cast("int").alias("edad"),
            F.upper(F.trim("tipo_edad")).alias("tipo_edad"),
            F.upper(F.trim("sexo")).alias("sexo"),
        )
        .withColumn("cod_dep", F.substring("ubigeo", 1, 2))
        .withColumn("edad_anios",
            F.when(F.col("tipo_edad") == "A", F.col("edad"))
             .when(F.col("tipo_edad") == "M", F.floor(F.col("edad") / 12))
             .when(F.col("tipo_edad") == "D", F.floor(F.col("edad") / 365)))
    )
    semana_valida = (F.col("semana").between(1, 52)
                     | ((F.col("semana") == 53) & F.col("ano").isin(ANIOS_53)))
    reglas = {
        "anio_fuera_de_rango": ~F.col("ano").between(ANIO_INICIO, ANIO_FIN),
        "semana_inexistente":  ~semana_valida,
        "ubigeo_invalido":     ~F.col("ubigeo").rlike("^[0-9]{6}$") | ~F.col("cod_dep").between("01", "25"),
        "sexo_invalido":       ~F.col("sexo").isin("F", "M"),
        "edad_invalida":       ~F.col("edad_anios").between(0, 110),
        "no_es_dengue":        ~F.col("diagnostic").isin("A97.0", "A97.1", "A97.2"),
    }
    motivo = F.concat_ws(",", *[F.when(F.coalesce(cond, F.lit(True)), F.lit(nombre))
                                for nombre, cond in reglas.items()])
    return df.withColumn("motivo_rechazo", motivo)


df_bronze = spark.read.parquet(ruta_salida("bronze", "dengue"))
if USAR_CACHE:
    df_bronze = df_bronze.cache()
df_eval = transformar_dengue_silver(df_bronze)

with cronometro("dengue_calidad"):
    total = df_bronze.count()
    columnas_origen = [c for c in df_bronze.columns if not c.startswith("_")]
    duplicados = total - df_bronze.select(columnas_origen).distinct().count()
    print(f"Registros en Bronze: {total}")
    print(f"Filas idénticas: {duplicados} (se conservan: sin ID de caso pueden ser pacientes distintos)")

    criticos = ["departamento", "ubigeo", "ano", "semana", "diagnostic", "edad", "tipo_edad", "sexo"]
    print("Completitud por campo crítico (%):")
    df_bronze.select([F.round(F.avg(F.when(F.trim(c) != "", 1).otherwise(0)) * 100, 2).alias(c)
                      for c in criticos]).show(vertical=True)

    print("Resultado de las reglas ('' = válido):")
    df_eval.groupBy("motivo_rechazo").count().orderBy(F.desc("count")).show(truncate=False)

with cronometro("dengue_silver_escritura"):
    (df_eval.filter(F.col("motivo_rechazo") == "").drop("motivo_rechazo")
        .repartition("ano").write.mode("overwrite").partitionBy("ano")
        .parquet(ruta_salida("silver", "dengue_nacional")))
    (df_eval.filter(F.col("motivo_rechazo") != "")
        .coalesce(1).write.mode("overwrite")
        .parquet(ruta_salida("silver", "dengue_rechazados")))

if USAR_CACHE:
    df_bronze.unpersist()

Registros en Bronze: 1029421
Filas idénticas: 152091 (se conservan: sin ID de caso pueden ser pacientes distintos)
Completitud por campo crítico (%):
-RECORD 0-------------
 departamento | 100.0 
 ubigeo       | 100.0 
 ano          | 100.0 
 semana       | 100.0 
 diagnostic   | 100.0 
 edad         | 100.0 
 tipo_edad    | 100.0 
 sexo         | 100.0 

Resultado de las reglas ('' = válido):
+------------------+-------+
|motivo_rechazo    |count  |
+------------------+-------+
|                  |1029326|
|semana_inexistente|81     |
|edad_invalida     |14     |
+------------------+-------+

[dengue_calidad] 5.48 s
[dengue_silver_escritura] 9.37 s


### 4. Silver: clima semanal por semana epidemiológica

In [5]:
def transformar_clima_silver(df_bronze_clima):
    variables = ["T2M", "RH2M", "PRECTOTCORR"]
    diario = (
        df_bronze_clima.select(
            F.trim("cod_dep").alias("cod_dep"),
            F.to_date("fecha", "yyyyMMdd").alias("fecha"),
            # -999 es el valor de relleno de NASA POWER: se convierte en nulo
            *[F.when(F.col(v).cast("double") == -999, None).otherwise(F.col(v).cast("double")).alias(v)
              for v in variables])
        # Rangos físicos: un valor fuera de rango se anula (no se elimina el día)
        .withColumn("T2M", F.when(F.col("T2M").between(-10, 45), F.col("T2M")))
        .withColumn("RH2M", F.when(F.col("RH2M").between(0, 100), F.col("RH2M")))
        .withColumn("PRECTOTCORR", F.when(F.col("PRECTOTCORR") >= 0, F.col("PRECTOTCORR")))
        # Semana epidemiológica: el miércoles de cada semana (domingo a sábado) define ano y semana
        .withColumn("miercoles", F.expr("date_add(fecha, 4 - dayofweek(fecha))"))
        .withColumn("ano", F.year("miercoles"))
        .withColumn("semana", (F.floor((F.dayofyear("miercoles") - 1) / 7) + 1).cast("int"))
    )
    return diario


df_diario = transformar_clima_silver(spark.read.parquet(ruta_salida("bronze", "clima"))).cache()

with cronometro("clima_calidad"):
    print("Valores nulos tras las reglas (relleno -999 o fuera de rango):")
    df_diario.select([F.sum(F.col(v).isNull().cast("int")).alias(v)
                      for v in ["fecha", "T2M", "RH2M", "PRECTOTCORR"]]).show()
    print("Días con precipitación > 50 mm (se conservan, marcados como atípicos):")
    df_diario.filter(F.col("PRECTOTCORR") > 50).groupBy("cod_dep").count().orderBy("cod_dep").show()

clima_semanal = (
    df_diario.groupBy("cod_dep", "ano", "semana")
    .agg(F.count("*").alias("n_dias"),
         F.round(F.avg("T2M"), 2).alias("temp_media"),
         F.round(F.avg("RH2M"), 2).alias("hum_media"),
         F.round(F.sum("PRECTOTCORR"), 2).alias("precip_total"))
    .filter(F.col("ano").between(ANIO_INICIO, ANIO_FIN) & (F.col("n_dias") == 7))
)

with cronometro("clima_silver_escritura"):
    clima_semanal.coalesce(1).write.mode("overwrite").parquet(ruta_salida("silver", "clima_semanal"))

clima_semanal = spark.read.parquet(ruta_salida("silver", "clima_semanal"))
semanas_por_dep = clima_semanal.groupBy("cod_dep").count().select("count").distinct().collect()
anios_53_clima = sorted(r["ano"] for r in clima_semanal.filter("semana = 53").select("ano").distinct().collect())
print("Semanas por departamento:", [r["count"] for r in semanas_por_dep])
assert anios_53_clima == ANIOS_53, f"Calendario inconsistente: {anios_53_clima} vs {ANIOS_53}"
print("Calendario del clima coincide con el calendario epidemiológico del dengue")
df_diario.unpersist()

Valores nulos tras las reglas (relleno -999 o fuera de rango):
+-----+---+----+-----------+
|fecha|T2M|RH2M|PRECTOTCORR|
+-----+---+----+-----------+
|    0|  0|   0|          0|
+-----+---+----+-----------+

Días con precipitación > 50 mm (se conservan, marcados como atípicos):
+-------+-----+
|cod_dep|count|
+-------+-----+
|     01|    2|
|     05|    6|
|     06|    2|
|     08|    6|
|     10|   40|
|     12|    3|
|     13|    1|
|     16|  224|
|     17|  135|
|     19|    8|
|     20|    2|
|     22|   15|
|     24|    9|
|     25|   39|
+-------+-----+

[clima_calidad] 1.10 s
[clima_silver_escritura] 1.85 s
Semanas por departamento: [1304]
Calendario del clima coincide con el calendario epidemiológico del dengue


DataFrame[cod_dep: string, fecha: date, T2M: double, RH2M: double, PRECTOTCORR: double, miercoles: date, ano: int, semana: int]

### 5. Gold: panel departamento × semana epidemiológica

In [6]:
MIN_CASOS_DEPARTAMENTO = 1000   # Por debajo, la serie semanal es casi toda ceros y no hay canal endémico

def construir_panel(df_dengue, df_clima, min_casos):
    casos_dep = df_dengue.groupBy("cod_dep", "departamento").agg(F.count("*").alias("casos_total"))
    deps = casos_dep.filter(F.col("casos_total") >= min_casos).select("cod_dep", "departamento")
    calendario = spark.createDataFrame(
        [(a, s) for a in range(ANIO_INICIO, ANIO_FIN + 1)
                for s in range(1, (53 if a in ANIOS_53 else 52) + 1)],
        ["ano", "semana"])
    casos = df_dengue.groupBy("cod_dep", "ano", "semana").agg(F.count("*").alias("casos"))
    panel = (deps.crossJoin(calendario)                                   # todas las semanas, con o sin casos
             .join(casos, ["cod_dep", "ano", "semana"], "left")
             .fillna(0, subset=["casos"])
             .join(df_clima.drop("n_dias"), ["cod_dep", "ano", "semana"], "left"))
    return panel, casos_dep


df_dengue_silver = spark.read.parquet(ruta_salida("silver", "dengue_nacional"))
panel, casos_dep = construir_panel(df_dengue_silver, clima_semanal, MIN_CASOS_DEPARTAMENTO)

with cronometro("gold_escritura"):
    (panel.select("cod_dep", "departamento", "ano", "semana", "casos",
                  "temp_media", "hum_media", "precip_total")
          .orderBy("cod_dep", "ano", "semana")
          .coalesce(1).write.mode("overwrite").parquet(ruta_salida("gold", "panel_dengue_clima")))

# Controles de Gold: cada uno es evidencia para la sección 3.4
gold = spark.read.parquet(ruta_salida("gold", "panel_dengue_clima"))
n_deps = gold.select("cod_dep").distinct().count()
semanas_esperadas = 25 * 52 + len(ANIOS_53)
filas = gold.count()
casos_gold = gold.agg(F.sum("casos")).first()[0]
casos_silver = casos_dep.filter(F.col("casos_total") >= MIN_CASOS_DEPARTAMENTO).agg(F.sum("casos_total")).first()[0]
nulos_clima = gold.filter(F.col("temp_media").isNull() | F.col("precip_total").isNull()).count()

print("Excluidos por baja transmisión:")
casos_dep.filter(F.col("casos_total") < MIN_CASOS_DEPARTAMENTO).orderBy(F.desc("casos_total")).show()
print(f"Departamentos: {n_deps} | filas: {filas} (esperadas {n_deps * semanas_esperadas})")
print(f"Casos en Gold: {casos_gold} | casos en Silver de esos departamentos: {casos_silver}")
print(f"Semanas sin clima: {nulos_clima}")
assert filas == n_deps * semanas_esperadas, "El panel tiene semanas faltantes o duplicadas"
assert casos_gold == casos_silver, "Se perdieron o duplicaron casos al agregar"
assert nulos_clima == 0, "Hay departamentos o semanas sin dato climático"

[gold_escritura] 3.74 s
Excluidos por baja transmisión:
+-------+------------+-----------+
|cod_dep|departamento|casos_total|
+-------+------------+-----------+
|     21|        PUNO|        647|
|     04|    AREQUIPA|         28|
|     03|    APURIMAC|         11|
|     18|    MOQUEGUA|          1|
+-------+------------+-----------+

Departamentos: 19 | filas: 24776 (esperadas 24776)
Casos en Gold: 1028639 | casos en Silver de esos departamentos: 1028639
Semanas sin clima: 0


### 6. Tiempos de esta corrida (evidencia para la sección 3.6)

In [7]:
df_tiempos = pd.DataFrame(tiempos)
df_tiempos.loc[len(df_tiempos)] = {"etapa": "TOTAL", "segundos": df_tiempos["segundos"].sum()}
display(df_tiempos)

dir_metricas = Path(ruta_salida("metricas", ""))
dir_metricas.mkdir(parents=True, exist_ok=True)
df_tiempos.to_csv(dir_metricas / f"tiempos_cache{USAR_CACHE}_sp{SHUFFLE_PARTITIONS}.csv", index=False)

,etapa,segundos
0,bronze_dengue,12.13
1,bronze_clima,2.59
2,dengue_calidad,5.48
3,dengue_silver_escritura,9.37
4,clima_calidad,1.10
5,clima_silver_escritura,1.85
6,gold_escritura,3.74
7,TOTAL,36.26


## 7. Prueba de escalabilidad (sección 3.7)
Los datos replicados se usan SOLO para medir tiempos. No entran al análisis ni al modelo.
Se escriben en data/escalabilidad/ y se borran después.

In [8]:
EJECUTAR_ESCALABILIDAD = True
FACTORES = [1, 10, 100]   # Prueben primero con [1, 10]; ×100 son ~100 millones de filas
resultados = []
df_base = spark.read.parquet(ruta_salida("bronze", "dengue"))

for n in FACTORES:
    ruta_rep = ruta_salida("escalabilidad", f"bronze_x{n}")
    (df_base.crossJoin(spark.range(n).withColumnRenamed("id", "_replica"))
            .write.mode("overwrite").parquet(ruta_rep))          # generar la réplica no se mide
    registros = spark.read.parquet(ruta_rep).count()

    inicio = time.perf_counter()
    (transformar_dengue_silver(spark.read.parquet(ruta_rep))
        .filter(F.col("motivo_rechazo") == "").drop("motivo_rechazo")
        .write.mode("overwrite").parquet(ruta_salida("escalabilidad", f"silver_x{n}")))
    panel_rep, _ = construir_panel(spark.read.parquet(ruta_salida("escalabilidad", f"silver_x{n}")),
                                   clima_semanal, MIN_CASOS_DEPARTAMENTO * n)
    filas_panel = panel_rep.count()
    segundos = time.perf_counter() - inicio

    resultados.append({"factor": n, "registros": registros,
                       "segundos": round(segundos, 1), "filas_panel": filas_panel})
    print(resultados[-1])

df_esc = pd.DataFrame(resultados)
df_esc["tiempo_relativo"] = (df_esc["segundos"] / df_esc["segundos"].iloc[0]).round(1)
display(df_esc)
df_esc.to_csv(dir_metricas / "escalabilidad.csv", index=False)

{'factor': 1, 'registros': 1029421, 'segundos': 9.1, 'filas_panel': 24776}
{'factor': 10, 'registros': 10294210, 'segundos': 11.5, 'filas_panel': 24776}
{'factor': 100, 'registros': 102942100, 'segundos': 45.3, 'filas_panel': 24776}


,factor,registros,segundos,filas_panel,tiempo_relativo
0,1,1029421,9.1,24776,1.0
1,10,10294210,11.5,24776,1.3
2,100,102942100,45.3,24776,5.0
